In [9]:
"""
African Exports Analysis
=======================
This script analyzes transport costs and logistics patterns specifically for 
African countries' exports, combining predictive modeling and anomaly detection.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
from category_encoders import TargetEncoder
import shap
import os
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

# Create output directories
output_dir = "african_exports"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# List of African countries (ISO3 codes)
african_countries = [
    "DZA", "AGO", "BEN", "BWA", "BFA", "BDI", "CMR", "CPV", "CAF", "TCD", "COM",
    "COD", "DJI", "EGY", "GNQ", "ERI", "ETH", "GAB", "GMB", "GHA", "GIN", "GNB",
    "CIV", "KEN", "LSO", "LBR", "LBY", "MDG", "MWI", "MLI", "MRT", "MUS", "MYT",
    "MAR", "MOZ", "NAM", "NER", "NGA", "REU", "RWA", "STP", "SEN", "SYC", "SLE",
    "SOM", "ZAF", "SSD", "SDN", "SWZ", "TZA", "TGO", "TUN", "UGA", "ESH", "ZMB", "ZWE"
]

print("Starting African Exports Analysis...")

# Load the dataset
try:
    df = pd.read_csv("imputed_full_matrix_at_centroid.csv")
except FileNotFoundError:
    df = pd.read_csv("../../../../../data/imputed_full_matrix_at_centroid.csv")

print(f"Full dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns")

# Filter for exports from African countries only
df_africa = df[df['origin_ISO'].isin(african_countries)]
print(f"African exports dataset: {df_africa.shape[0]} rows ({df_africa.shape[0]/df.shape[0]:.1%} of total data)")

#############################
# 1. OVERVIEW OF AFRICAN EXPORTS
#############################
print("\n\n===== OVERVIEW OF AFRICAN EXPORTS =====")

# Top exporting African countries by volume
top_exporters_volume = df_africa.groupby('origin_ISO')['flow(tonne)'].sum().sort_values(ascending=False)
print("\nTop 10 African Exporters by Volume (tonnes):")
print(top_exporters_volume.head(10))

# Top exporting African countries by trade routes
top_exporters_routes = df_africa.groupby('origin_ISO').size().sort_values(ascending=False)
print("\nTop 10 African Exporters by Number of Trade Routes:")
print(top_exporters_routes.head(10))

# Export costs by country
export_costs = df_africa.groupby('origin_ISO')['Unit logistics costs ($/ton)'].agg(['mean', 'median', 'min', 'max', 'count']).sort_values('mean', ascending=False)
print("\nAfrican Countries by Average Export Costs ($/ton):")
print(export_costs.head(10))
print("\nAfrican Countries with Lowest Export Costs ($/ton):")
print(export_costs.tail(10))

# Visualization - Export costs by country
plt.figure(figsize=(14, 8))
export_costs_plot = export_costs.sort_values('mean')
plt.bar(export_costs_plot.index, export_costs_plot['mean'], yerr=export_costs_plot['mean'].std())
plt.title('Average Export Costs by African Country', fontsize=16)
plt.xlabel('Country', fontsize=14)
plt.ylabel('Average Unit Logistics Costs ($/ton)', fontsize=14)
plt.xticks(rotation=90)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(f"{output_dir}/african_export_costs.png")
plt.close()

# Transport mode distribution for African exports
mode_dist = pd.crosstab(df_africa['origin_ISO'], df_africa['Mode_name'], normalize='index') * 100
print("\nTransport Mode Distribution by Country (%):")
print(mode_dist.head(10))

# Overall mode distribution
overall_mode_dist = df_africa['Mode_name'].value_counts(normalize=True) * 100
print("\nOverall Transport Mode Distribution for African Exports:")
print(overall_mode_dist)

# Visualization - Mode distribution
plt.figure(figsize=(10, 6))
overall_mode_dist.plot(kind='bar', color='darkblue')
plt.title('Transport Mode Distribution for African Exports', fontsize=16)
plt.xlabel('Transport Mode', fontsize=14)
plt.ylabel('Percentage of Exports (%)', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(f"{output_dir}/african_mode_distribution.png")
plt.close()

# Top export destinations
top_destinations = df_africa.groupby('destination_ISO').agg({
    'flow(tonne)': 'sum',
    'Unit logistics costs ($/ton)': 'mean'
}).sort_values('flow(tonne)', ascending=False)
print("\nTop 10 Destinations for African Exports:")
print(top_destinations.head(10))

# Destination regions
def get_region(country_code):
    regions = {
        'Africa': african_countries,
        'Europe': ['DEU', 'FRA', 'GBR', 'ITA', 'ESP', 'NLD', 'BEL', 'CHE', 'AUT', 'PRT', 'SWE', 'NOR', 'FIN', 'DNK', 'IRL', 'LUX'],
        'North America': ['USA', 'CAN', 'MEX'],
        'South America': ['BRA', 'ARG', 'CHL', 'COL', 'PER', 'VEN', 'ECU', 'BOL', 'PRY', 'URY'],
        'Asia': ['CHN', 'JPN', 'KOR', 'IND', 'IDN', 'THA', 'MYS', 'SGP', 'VNM', 'PHL', 'PAK', 'BGD', 'SAU', 'ARE', 'IRN', 'TUR'],
        'Oceania': ['AUS', 'NZL', 'PNG', 'FJI']
    }
    
    for region, countries in regions.items():
        if country_code in countries:
            return region
    return 'Other'

df_africa['destination_region'] = df_africa['destination_ISO'].apply(get_region)
region_dist = df_africa.groupby('destination_region').agg({
    'flow(tonne)': 'sum',
    'Unit logistics costs ($/ton)': 'mean'
}).sort_values('flow(tonne)', ascending=False)

print("\nAfrican Exports by Destination Region:")
print(region_dist)

# Visualization - Exports by destination region
plt.figure(figsize=(12, 6))
fig, ax1 = plt.subplots(figsize=(12, 6))

x = region_dist.index
y1 = region_dist['flow(tonne)'] / 1_000_000  # Convert to millions of tonnes

ax1.bar(x, y1, color='blue', alpha=0.7)
ax1.set_xlabel('Destination Region', fontsize=14)
ax1.set_ylabel('Total Volume (Million Tonnes)', color='blue', fontsize=14)
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_xticklabels(x, rotation=45)

ax2 = ax1.twinx()
y2 = region_dist['Unit logistics costs ($/ton)']
ax2.plot(x, y2, 'ro-', linewidth=2, markersize=8)
ax2.set_ylabel('Average Logistics Cost ($/ton)', color='red', fontsize=14)
ax2.tick_params(axis='y', labelcolor='red')

plt.title('African Exports by Destination Region: Volume and Cost', fontsize=16)
plt.tight_layout()
plt.savefig(f"{output_dir}/african_exports_by_region.png")
plt.close()

# Top commodities
top_commodities = df_africa.groupby('IFM_HS').agg({
    'flow(tonne)': 'sum',
    'Unit logistics costs ($/ton)': 'mean'
}).sort_values('flow(tonne)', ascending=False)
print("\nTop 10 Export Commodities from Africa:")
print(top_commodities.head(10))

#############################
# 2. TRANSPORT COST PREDICTION MODEL FOR AFRICAN EXPORTS
#############################
print("\n\n===== TRANSPORT COST PREDICTION MODEL FOR AFRICAN EXPORTS =====")

# Remove extreme outliers for modeling purposes
q1 = df_africa['Unit logistics costs ($/ton)'].quantile(0.01)
q3 = df_africa['Unit logistics costs ($/ton)'].quantile(0.99)
df_africa_model = df_africa[(df_africa['Unit logistics costs ($/ton)'] >= q1) & 
                           (df_africa['Unit logistics costs ($/ton)'] <= q3)]

print(f"Removed {df_africa.shape[0] - df_africa_model.shape[0]} outliers for modeling")

# Define features for modeling
numeric_features = ['distance(km)', 'flow(tonne)']
categorical_features = ['Mode_name', 'destination_ISO', 'IFM_HS', 'ship_type', 'container_type']

# Prepare features and target variable
X = df_africa_model[numeric_features + categorical_features].copy()
y = df_africa_model['Unit logistics costs ($/ton)']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

# Create preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('target_encoder', TargetEncoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Define the model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(random_state=42))
])

# Train the model
print("\nTraining XGBoost model for African export costs...")
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model performance:")
print(f"  RMSE: {rmse:.2f}")
print(f"  MAE: {mae:.2f}")
print(f"  R²: {r2:.4f}")

Starting African Exports Analysis...
Full dataset loaded with 9512578 rows and 15 columns
African exports dataset: 1579128 rows (16.6% of total data)


===== OVERVIEW OF AFRICAN EXPORTS =====

Top 10 African Exporters by Volume (tonnes):
origin_ISO
ZAF    2.197497e+08
NGA    1.596314e+08
EGY    1.061432e+08
DZA    6.224474e+07
SDN    3.580746e+07
ETH    3.273235e+07
MAR    3.251978e+07
KEN    2.707543e+07
TZA    2.394440e+07
AGO    2.332599e+07
Name: flow(tonne), dtype: float64

Top 10 African Exporters by Number of Trade Routes:
origin_ISO
ZAF    103816
MAR     52212
KEN     52212
TUN     52212
CMR     52212
NGA     52212
DZA     52212
SEN     26182
MOZ     26182
MRT     26182
dtype: int64

African Countries by Average Export Costs ($/ton):
                    mean       median       min            max  count
origin_ISO                                                           
MUS         17654.140096  6521.706543  0.992855  149686.984252  16340
COM         17051.693625  5888.454134 

<Figure size 1200x600 with 0 Axes>

In [11]:
# Add hyperparameter tuning
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.8, 1.0],
    'model__min_child_weight': [1, 3]
}

# Use RandomizedSearchCV for faster tuning

# Option 1: Disable parallel processing
from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    model, param_grid, n_iter=10, cv=3, 
    scoring='neg_root_mean_squared_error', 
    random_state=42, n_jobs=1  # Change to single job
)

random_search.fit(X_train, y_train)

# Get best model
best_model = random_search.best_estimator_
print(f"Best parameters: {random_search.best_params_}")

# Evaluate tuned model
y_pred = best_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Improved model performance:")
print(f"  RMSE: {rmse:.2f}")
print(f"  MAE: {mae:.2f}")
print(f"  R²: {r2:.4f}")

Best parameters: {'model__subsample': 0.8, 'model__n_estimators': 200, 'model__min_child_weight': 3, 'model__max_depth': 5, 'model__learning_rate': 0.1}
Improved model performance:
  RMSE: 4708.62
  MAE: 1895.52
  R²: 0.8286
